# 06 — End-to-end dry run of a full session

Replays the whole fixture day through the production runner (`runner.run_session`)
on a controllable clock, with injected faults.

**Expected behaviour**
- nothing runs before an explicit authorization; the token is consumed on start;
- exactly one analysis per symbol per completed 15-minute candle, no duplicates;
- a late candle is skipped (no request), a timeout is retried then logged, an invalid
  reply is rejected;
- an unexpected crash mid-session does not resume until re-authorized;
- the session shuts down at the close with the runtime flag reset;
- no brokerage module is loaded and no order is ever submitted.

**Modes.** `MOCK` (default) makes zero external LLM calls. `REAL_LLM` needs
`QBS_AGENT_TEST_MODE=REAL_LLM` **and** `QBS_AGENT_TEST_ALLOW_PAID=1`, prints a cost
estimate first, and stops at `QBS_AGENT_TEST_MAX_CALLS` (default 20).

**Isolation.** Every run uses a fresh temporary database (`tk.workspace()`); the
production config (`var/agent/config.json`) and log (`var/agent/agent.db`) are never
read or written, and nothing here can reach `qbs.live` or a broker.

**Dependencies.** `pip install -r requirements.txt jupyter` (MOCK). REAL_LLM also needs
the selected provider's package and key (see docs/TRADING_AGENT.md).

In [1]:
import os, sys, pathlib, json, logging
here = pathlib.Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "qbs" / "trading_agent").is_dir())
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.WARNING)

import pandas as pd
from qbs.trading_agent import testkit as tk

MODE = tk.test_mode()          # MOCK unless QBS_AGENT_TEST_MODE=REAL_LLM and QBS_AGENT_TEST_ALLOW_PAID=1
print("repo:", ROOT)
print("MODE:", MODE, "| paid-call cap:", tk.max_calls() if MODE == tk.REAL_LLM else 0)

repo: /home/user/qqq_boxx_strategies
MODE: MOCK | paid-call cap: 0


In [2]:
import time, threading
from qbs.trading_agent import session as ss, store, evaluation
from qbs.trading_agent.runner import run_session
from qbs.trading_agent.engine import TradingAgentEngine
from qbs.trading_agent.market_context import MarketContextBuilder
from qbs.trading_agent.session_calendar import analysis_slots

fx = tk.load_fixture(); day = fx.replay_date
check = tk.Check("06 end-to-end")
modules_before = set(sys.modules)
live_db = ROOT / "var" / "live.db"
live_before = live_db.stat().st_mtime if live_db.exists() else None
ws = tk.workspace(MODE, max_retries=1, request_timeout_s=0.5 if MODE == tk.MOCK else 60)
cfg = ws.cfg
slots = analysis_slots(day)
if MODE == tk.REAL_LLM:
    print("ABOUT TO SPEND (full day, capped):", tk.cost_preview(cfg, min(tk.max_calls(), 3 * len(slots))))

## Fault plan

In [3]:
STALE_AT = tk.et(day, 10, 30)          # TEAM's 10:30 candle arrives late
TIMEOUT_AT = tk.et(day, 11, 0)         # every request at 11:00 hangs past the timeout
INVALID_AT = tk.et(day, 12, 0)         # the model answers prose at 12:00
CRASH_AT = tk.et(day, 13, 0)           # the process dies after the 13:00 cycle

base = tk.make_provider(MODE, cfg)
class Faulty:
    """Wraps the provider and injects faults by candle. Production code is untouched."""
    name = "faulty"
    def __init__(self, inner): self.inner, self.calls = inner, []
    def complete(self, system, user, settings):
        self.calls.append(user)
        if TIMEOUT_AT.isoformat() in user:
            time.sleep(settings.timeout_s + 0.2)
        if INVALID_AT.isoformat() in user:
            return tk.response_with_usage("Sorry, I think it's a buy.", 1000, 10)
        return self.inner.complete(system, user, settings)
llm = Faulty(base)
pd.DataFrame({"fault": ["stale", "timeout", "invalid", "crash"],
              "at": [STALE_AT, TIMEOUT_AT, INVALID_AT, CRASH_AT]})

,fault,at
0,stale,2026-10-08 10:30:00-04:00
1,timeout,2026-10-08 11:00:00-04:00
2,invalid,2026-10-08 12:00:00-04:00
3,crash,2026-10-08 13:00:00-04:00


## Before authorization nothing runs

In [4]:
clock = tk.Clock(tk.et(day, 9, 0))
ss.save_daily_prompt(cfg, fx.prompt, clock())
feed = tk.FixtureBars(fx, clock, withhold={("TEAM", STALE_AT.isoformat())})
clock.set(tk.et(day, 9, 31))
s = run_session(cfg, llm, feed, clock=clock, sleep=clock.sleep, install_signals=False)
check("no authorization -> no session", s is None)
check("...and no LLM call, no data fetch", not llm.calls and feed.calls == 0)
cfg.agent_enabled = True                    # the runner resets this on every exit
ok, msg = ss.authorize(cfg, clock()); print(msg)

authorized 2026-10-08, prompt v1


## Morning session until the crash

The production runner owns the morning. At the crash point the engine is abandoned without a shutdown (a real `kill -9` skips every `finally`), so this part drives `engine.run_cycle` slot by slot after `activate` — the same calls the runner makes.

In [5]:
sess, why = ss.activate(cfg, clock()); sess.transition(ss.RUNNING, "nb06 morning")
eng = TradingAgentEngine(cfg, sess, llm, MarketContextBuilder(feed, cfg.market_context),
                         sleep=clock.sleep)
for slot in [s for s in slots if s <= CRASH_AT]:
    clock.set(slot + pd.Timedelta(seconds=cfg.data_delay_seconds + 5))
    eng.run_cycle(slot)
# --- crash: no transition, no shutdown, the objects are simply gone ---
crashed_id = sess.session_id
del eng, sess

## Restart: refused until re-authorized, then the runner finishes the day

In [6]:
clock.set(CRASH_AT + pd.Timedelta(minutes=7))
cfg.agent_enabled = True
s = run_session(cfg, llm, feed, clock=clock, sleep=clock.sleep, install_signals=False)
check("restart without a new token stays disabled", s is None)
row = store.one(cfg.db_path, "SELECT state, agent_enabled, close_reason FROM sessions WHERE session_id = ?", (crashed_id,))
check("crashed session closed as ERROR", row["state"] == "ERROR" and row["agent_enabled"] == 0, row["close_reason"])
cfg.agent_enabled = True
ss.authorize(cfg, clock())
s = run_session(cfg, llm, feed, clock=clock, sleep=clock.sleep, install_signals=False)
check("afternoon session ran to the close", s is not None and s.state == "STOPPED")
final = store.one(cfg.db_path, "SELECT * FROM sessions WHERE session_id = ?", (s.session_id,))
check("closed at end of session, flag reset", "end of regular session" in final["close_reason"] and final["agent_enabled"] == 0)
check("runtime agent_enabled reset", cfg.agent_enabled is False)

True

## Scheduling and fault outcomes

In [7]:
cyc = pd.DataFrame(store.rows(cfg.db_path, "SELECT session_date, symbol, candle_ts, status FROM cycles"))
check("no duplicate (symbol, candle)", not cyc.duplicated(["symbol", "candle_ts"]).any())
analysed = set(cyc.candle_ts)
expected = {s.isoformat() for s in slots}
check("every slot analysed exactly once across the crash", analysed == expected,
      f"missing {sorted(expected - analysed)[:3]}")
by = cyc.set_index(["symbol", "candle_ts"]).status
check("late candle skipped", by[("TEAM", STALE_AT.isoformat())] == "SKIPPED_STALE")
check("timeouts logged as LLM_ERROR", (cyc[cyc.candle_ts == TIMEOUT_AT.isoformat()].status == "LLM_ERROR").all())
check("prose rejected", (cyc[cyc.candle_ts == INVALID_AT.isoformat()].status == "INVALID_SCHEMA").all())
req = pd.DataFrame(store.requests_for(cfg.db_path))
t = req[req.candle_ts == TIMEOUT_AT.isoformat()]
check("timeout retried once, counted", (t.retry_count == 1).all() and (t.timeout_events == 2).all())
check("no request for the stale candle", not ((req.symbol == "TEAM") & (req.candle_ts == STALE_AT.isoformat())).any())
cyc.status.value_counts()

status
VALID             68
LLM_ERROR          3
INVALID_SCHEMA     3
SKIPPED_STALE      1
Name: count, dtype: int64

## No brokerage access, ever

In [8]:
loaded = tk.broker_modules_loaded(modules_before)
check("no broker / qbs.live module imported", not loaded, ", ".join(loaded))
tables = {r["name"] for r in store.rows(cfg.db_path, "SELECT name FROM sqlite_master WHERE type='table'")}
check("agent DB has no order tables", not any("order" in t or "fill" in t for t in tables))
live_after = live_db.stat().st_mtime if live_db.exists() else None
check("the live trading log was not created or touched", live_after == live_before)
check("every call was DRY_RUN", all('"execution_mode": "DRY_RUN"' in u for u in llm.calls))

True

## Evaluation summary

In [9]:
summary = {"mode": MODE, "fixture": fx.version, "replay_date": str(day),
           "operational": evaluation.operational(cfg.db_path),
           "decisions": evaluation.decisions(cfg.db_path),
           "checks": check.rows}
print(json.dumps({k: summary["operational"][k] for k in ("analysis_cycles", "cycle_status", "llm_requests",
                  "success_rate", "validation_pass_rate", "timeouts", "retries")}, indent=1))
print(json.dumps(summary["decisions"], indent=1))
print(tk.save_results("06_end_to_end_summary", summary))
display(check.frame()); check.raise_if_failed()

{
 "analysis_cycles": 75,
 "cycle_status": {
  "VALID": 68,
  "SKIPPED_STALE": 1,
  "LLM_ERROR": 3,
  "INVALID_SCHEMA": 3
 },
 "llm_requests": 74,
 "success_rate": 0.9594594594594594,
 "validation_pass_rate": 0.9577464788732394,
 "timeouts": 6,
 "retries": 3
}
{
 "decisions": 75,
 "status": {
  "VALID": 68,
  "LLM_ERROR": 3,
  "INVALID_SCHEMA": 3,
  "SKIPPED_STALE": 1
 },
 "action_distribution": {
  "HOLD": 24,
  "SELL": 1,
  "NO_TRADE": 41,
  "BUY": 2
 },
 "consecutive_pairs": 65,
 "decision_change_rate": 0.12307692307692308,
 "buy_sell_flip_rate": 0.0,
 "compliance_rejections": 0,
 "schema_rejections": 3,
 "price_far_from_market": 0
}
/home/user/qqq_boxx_strategies/notebooks/agent_tests/results/06_end_to_end_summary_20261009T093339Z.json


,check,result,detail
0,no authorization -> no session,PASS,
1,"...and no LLM call, no data fetch",PASS,
2,restart without a new token stays disabled,PASS,
3,crashed session closed as ERROR,PASS,crash recovery: the process ended without a sh...
4,afternoon session ran to the close,PASS,
5,"closed at end of session, flag reset",PASS,
6,runtime agent_enabled reset,PASS,
7,"no duplicate (symbol, candle)",PASS,
8,every slot analysed exactly once across the crash,PASS,missing []
9,late candle skipped,PASS,


**Deployment gate.** Phase 1 is not deployed until this notebook, 02 and 04 pass
in MOCK mode and `pytest tests/test_trading_agent.py tests/test_agent_notebooks.py`
is green.

**Troubleshooting**
- *A slot missing*: the fake clock jumped past a slot + delay — check `data_delay_seconds`.
- *Timeout not retried*: `max_retries` on the workspace config is 0.
- *Broker module loaded*: an import was added to `qbs/trading_agent` — a release blocker.